# Stage 1 Baselines: Attention U-Net, U-Net++, ResUNet++ vs. FAB-UNet

Lightweight CNN baselines for the carotid-plaque segmentation comparison table. All three
reuse the **exact same** data pipeline, preprocessing, augmentation, loss function
(`combined_loss` = weighted BCE w/ `POS_WEIGHT` + Dice + Focal-Tversky + ramped boundary loss),
metrics, and training protocol (`train_one_fold`, checkpoint-selection on `val_dice`) as the
FAB-UNet notebook, so any performance difference reflects the **architecture only**.

No architecture here needs internet / pretrained weights, so this notebook is meant to be
cheap and fast to run end-to-end in a single Kaggle session.

**Architectures covered and why each is a relevant comparison:**
- **Attention U-Net** (Oktay et al., 2018) — tests whether attention gates alone (without
  FAB-UNet's spectral-attention / boundary-aware mechanism) already explain FAB-UNet's gains.
- **U-Net++** (Zhou et al., 2018) — nested, densely-connected skip pathways; a strong,
  commonly-cited baseline in medical image segmentation papers.
- **ResUNet++** (Jha et al., 2019) — residual blocks + squeeze-and-excitation + an ASPP
  bridge + attention-gated decoder; a directly comparable "complementary idea" to FAB-UNet.

Fill in your own FAB-UNet test-set numbers in the **reference row** cell near the end so the
final comparison table includes FAB-UNet for context without needing to retrain it here.

## 0-4. Shared pipeline (identical to the FAB-UNet notebook)

In [1]:
# ------------------------------------------------------------------
# 0. Package installation (safe no-ops if already present, as on Kaggle)
# ------------------------------------------------------------------
import subprocess, sys, importlib

def _ensure(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

for _pkg, _imp in [("opencv-python-headless", "cv2"), ("albumentations", "albumentations"),
                    ("seaborn", "seaborn"), ("pandas", "pandas"), ("scikit-learn", "sklearn"),
                    ("scipy", "scipy"), ("tqdm", "tqdm"), ("timm", "timm")]:
    _ensure(_pkg, _imp)
print("Dependency check complete.")

Dependency check complete.


In [2]:
import os, glob, random, json, math, time, warnings
from pathlib import Path
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

import torchvision
import timm
from torchvision.models._utils import IntermediateLayerGetter

from sklearn.model_selection import KFold, StratifiedKFold, train_test_split
from scipy.ndimage import distance_transform_edt
from tqdm.auto import tqdm

try:
    import albumentations as A
    HAS_ALBUMENTATIONS = True
except ImportError:
    HAS_ALBUMENTATIONS = False

warnings.filterwarnings("ignore")
print("Imports OK. Albumentations available:", HAS_ALBUMENTATIONS)

Imports OK. Albumentations available: True


In [3]:
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
if DEVICE.type == "cuda":
    print(torch.cuda.get_device_name(0))

Using device: cuda
Tesla T4


In [4]:
@dataclass
class CFG:
    DATA_ROOT: str = None
    TRAIN_IMAGE_DIR: str = None
    TRAIN_MASK_DIR: str = None
    TEST_IMAGE_DIR: str = None
    TEST_MASK_DIR: str = None

    PREP_ROOT: str = "/kaggle/working/preprocessed_384"
    OUTPUT_DIR: str = "/kaggle/working/outputs_stage1"

    IMG_SIZE: int = 384
    FORCE_REPREPROCESS: bool = False

    BATCH_SIZE: int = 6          # lower to 4 if CUDA OOM
    EPOCHS: int = 120
    LR: float = 1e-3
    K_FOLDS: int = 5
    EARLY_STOP_PATIENCE: int = 20
    NUM_WORKERS: int = 2
    MC_DROPOUT_P: float = 0.30
    MC_SAMPLES: int = 10

CFG = CFG()
CFG.DATA_ROOT = "/kaggle/input/datasets/chinnisiri/plaque-dataset/Plaque"

Path(CFG.PREP_ROOT).mkdir(parents=True, exist_ok=True)
Path(CFG.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print("Preprocessed data will be cached in:", CFG.PREP_ROOT)
print("All figures / models / csv reports will be saved in:", CFG.OUTPUT_DIR)

Preprocessed data will be cached in: /kaggle/working/preprocessed_384
All figures / models / csv reports will be saved in: /kaggle/working/outputs_stage1


In [5]:
import re

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
MASK_KEYWORDS  = ["mask", "masks", "gt", "ground_truth", "groundtruth", "label",
                  "labels", "segmentation", "annot"]
MARKED_KEYWORDS = ["marked", "annotated", "yellow"]
IMAGE_KEYWORDS = ["image", "images", "img", "raw", "original", "us_", "ultrasound"]

def _print_input_tree(base, max_depth=4):
    base = Path(base)
    print(f"Directory tree under {base} (for manual CFG.DATA_ROOT inspection):")
    for p in sorted(base.rglob("*")):
        if p.is_dir():
            depth = len(p.relative_to(base).parts)
            if depth <= max_depth:
                print("  " * depth + p.name + "/")

def _auto_detect_data_root():
    if CFG.DATA_ROOT is not None:
        return Path(CFG.DATA_ROOT)
    candidates = []
    search_base = Path("/kaggle/input")
    if search_base.exists():
        for p in search_base.rglob("*"):
            if p.is_dir():
                names = [d.name.lower() for d in p.iterdir() if d.is_dir()]
                has_train = any("train" in n for n in names)
                has_test  = any("test" in n for n in names)
                if has_train and has_test:
                    candidates.append(p)
    if candidates:
        candidates.sort(key=lambda p: len(p.parts))
        return candidates[0]
    if search_base.exists():
        _print_input_tree(search_base)
    raise FileNotFoundError(
        "Could not auto-detect a folder containing both train/ and test/ under "
        "/kaggle/input. Check the printed tree above and set CFG.DATA_ROOT manually "
        "to the dataset root, e.g. CFG.DATA_ROOT = '/kaggle/input/plaque-dataset/Plaque'."
    )

def _find_split_dir(data_root, split_name):
    for d in data_root.iterdir():
        if d.is_dir() and split_name in d.name.lower():
            return d
    raise FileNotFoundError(f"No '{split_name}' folder found under {data_root}")

def _find_role_dirs(split_dir, keywords):
    hits = []
    for d in [split_dir] + [x for x in split_dir.rglob("*") if x.is_dir()]:
        if any(k in d.name.lower() for k in keywords):
            hits.append(d)
    return hits

def _list_images(folder):
    files = [f for f in folder.rglob("*") if f.suffix.lower() in IMAGE_EXTS]
    return files

_MASK_SUFFIX_RE = re.compile(
    r"[_-](mask|masks|gt|ground[_-]?truth|label|labels|labeled|labelled|"
    r"segmentation|seg|annot|annotated|marked|binary)$"
)

def _stem_key(path):
    stem = path.stem.lower()
    m = _MASK_SUFFIX_RE.search(stem)
    if m:
        stem = stem[: m.start()]
    return stem

def discover_pairs(split_dir):
    mask_dirs = _find_role_dirs(split_dir, MASK_KEYWORDS)
    marked_dirs = _find_role_dirs(split_dir, MARKED_KEYWORDS)
    image_dirs = [d for d in _find_role_dirs(split_dir, IMAGE_KEYWORDS)
                  if d not in mask_dirs and d not in marked_dirs]
    mask_source_dirs = mask_dirs if mask_dirs else marked_dirs
    mask_is_marked = not mask_dirs and bool(marked_dirs)

    if image_dirs and mask_source_dirs and set(image_dirs) != set(mask_source_dirs):
        img_files = []
        for d in image_dirs:
            img_files += _list_images(d)
        mask_files = []
        for d in mask_source_dirs:
            mask_files += _list_images(d)
    else:
        all_files = _list_images(split_dir)
        mask_kw_all = MASK_KEYWORDS + MARKED_KEYWORDS
        mask_files = [f for f in all_files if any(k in f.stem.lower() for k in mask_kw_all)]
        mask_file_set = set(mask_files)
        img_files = [f for f in all_files if f not in mask_file_set]
        mask_is_marked = bool(mask_files) and not any(
            k in f.stem.lower() for f in mask_files for k in MASK_KEYWORDS
        )

    img_map = {_stem_key(p): p for p in img_files}
    mask_map = {_stem_key(p): p for p in mask_files}

    common = sorted(set(img_map) & set(mask_map))
    pairs = []
    for k in common:
        ip, mp = img_map[k], mask_map[k]
        if ip.resolve() == mp.resolve():
            continue
        pairs.append((ip, mp))
    return pairs, mask_is_marked

DATA_ROOT = _auto_detect_data_root()
print("Detected DATA_ROOT:", DATA_ROOT)

train_dir = Path(CFG.TRAIN_IMAGE_DIR).parent if CFG.TRAIN_IMAGE_DIR else _find_split_dir(DATA_ROOT, "train")
test_dir  = Path(CFG.TEST_IMAGE_DIR).parent  if CFG.TEST_IMAGE_DIR  else _find_split_dir(DATA_ROOT, "test")

train_pairs, train_marked = discover_pairs(train_dir)
test_pairs,  test_marked  = discover_pairs(test_dir)

print(f"Train pairs found: {len(train_pairs)}  (masks derived from 'marked' images: {train_marked})")
print(f"Test  pairs found: {len(test_pairs)}  (masks derived from 'marked' images: {test_marked})")
assert len(train_pairs) > 0 and len(test_pairs) > 0, (
    "No pairs discovered — please inspect the printed DATA_ROOT tree and set "
    "CFG.TRAIN_IMAGE_DIR / CFG.TRAIN_MASK_DIR / CFG.TEST_IMAGE_DIR / CFG.TEST_MASK_DIR manually."
)

Detected DATA_ROOT: /kaggle/input/datasets/chinnisiri/plaque-dataset/Plaque
Train pairs found: 493  (masks derived from 'marked' images: False)
Test  pairs found: 48  (masks derived from 'marked' images: False)


In [6]:
IMG_SIZE = CFG.IMG_SIZE

def resize_pad(img, size=IMG_SIZE, interp=cv2.INTER_AREA):
    h, w = img.shape[:2]
    scale = size / max(h, w)
    nh, nw = max(1, int(round(h * scale))), max(1, int(round(w * scale)))
    resized = cv2.resize(img, (nw, nh), interpolation=interp)
    pad_h, pad_w = size - nh, size - nw
    top, bottom = pad_h // 2, pad_h - pad_h // 2
    left, right = pad_w // 2, pad_w - pad_w // 2
    return cv2.copyMakeBorder(resized, top, bottom, left, right, cv2.BORDER_CONSTANT, value=0)

def preprocess_image(path):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Could not read image: {path}")
    img = resize_pad(img, IMG_SIZE, cv2.INTER_AREA)
    img = cv2.fastNlMeansDenoising(img, h=10, templateWindowSize=7, searchWindowSize=21)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img = clahe.apply(img)
    return img

def derive_mask_from_marked(marked_bgr):
    hsv = cv2.cvtColor(marked_bgr, cv2.COLOR_BGR2HSV)
    yellow = cv2.inRange(hsv, (15, 80, 80), (35, 255, 255))
    green  = cv2.inRange(hsv, (36, 40, 40), (85, 255, 255))
    annot = cv2.bitwise_or(yellow, green)
    annot = cv2.dilate(annot, np.ones((7, 7), np.uint8), iterations=2)
    contours, _ = cv2.findContours(annot, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mask = np.zeros(annot.shape, dtype=np.uint8)
    for c in contours:
        if cv2.contourArea(c) > 30:
            cv2.drawContours(mask, [c], -1, 255, thickness=cv2.FILLED)
    if mask.sum() == 0:
        mask = annot
    return mask

def preprocess_mask(mask_path, is_marked):
    raw = cv2.imread(str(mask_path))
    if raw is None:
        raise ValueError(f"Could not read mask: {mask_path}")
    if is_marked:
        binary = derive_mask_from_marked(raw)
    else:
        gray = cv2.cvtColor(raw, cv2.COLOR_BGR2GRAY) if raw.ndim == 3 else raw
        n_unique = len(np.unique(gray))
        if n_unique > 6:
            binary = derive_mask_from_marked(raw)
        else:
            binary = (gray > 127).astype(np.uint8) * 255
    binary = resize_pad(binary, IMG_SIZE, cv2.INTER_NEAREST)
    binary = (binary > 127).astype(np.uint8) * 255
    return binary

def run_preprocessing(pairs, split_name, is_marked, force=CFG.FORCE_REPREPROCESS):
    out_img_dir = Path(CFG.PREP_ROOT) / split_name / "images"
    out_mask_dir = Path(CFG.PREP_ROOT) / split_name / "masks"
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_mask_dir.mkdir(parents=True, exist_ok=True)

    manifest = []
    for idx, (img_path, mask_path) in enumerate(tqdm(pairs, desc=f"Preprocessing {split_name}")):
        out_name = f"{idx:04d}.png"
        out_img_path = out_img_dir / out_name
        out_mask_path = out_mask_dir / out_name
        if not force and out_img_path.exists() and out_mask_path.exists():
            manifest.append((str(img_path), str(mask_path), str(out_img_path), str(out_mask_path)))
            continue
        img_p = preprocess_image(img_path)
        mask_p = preprocess_mask(mask_path, is_marked)
        cv2.imwrite(str(out_img_path), img_p)
        cv2.imwrite(str(out_mask_path), mask_p)
        manifest.append((str(img_path), str(mask_path), str(out_img_path), str(out_mask_path)))

    df = pd.DataFrame(manifest, columns=["orig_image", "orig_mask", "prep_image", "prep_mask"])
    df.to_csv(Path(CFG.PREP_ROOT) / f"{split_name}_manifest.csv", index=False)
    return df

train_manifest = run_preprocessing(train_pairs, "train", train_marked)
test_manifest  = run_preprocessing(test_pairs, "test", test_marked)
print(f"Preprocessed {len(train_manifest)} train pairs and {len(test_manifest)} test pairs.")
print(f"Cached under: {CFG.PREP_ROOT}")

Preprocessing train:   0%|          | 0/493 [00:00<?, ?it/s]

Preprocessing test:   0%|          | 0/48 [00:00<?, ?it/s]

Preprocessed 493 train pairs and 48 test pairs.
Cached under: /kaggle/working/preprocessed_384


In [7]:
def build_train_augmentation():
    if not HAS_ALBUMENTATIONS:
        return None
    try:
        return A.Compose([
            A.HorizontalFlip(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.08, scale_limit=0.15, rotate_limit=12,
                               border_mode=cv2.BORDER_CONSTANT, p=0.6),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
            A.RandomGamma(gamma_limit=(80, 120), p=0.3),
            A.ElasticTransform(alpha=30, sigma=5, p=0.2),
            A.GaussNoise(p=0.2),
        ])
    except Exception as e:
        print("Falling back to the v1 augmentation pipeline:", e)
        return A.Compose([
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=15, border_mode=cv2.BORDER_CONSTANT, p=0.5),
            A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.4),
            A.GaussNoise(p=0.2),
        ])

def _fallback_augment(img, mask):
    if random.random() < 0.5:
        img, mask = np.fliplr(img).copy(), np.fliplr(mask).copy()
    if random.random() < 0.3:
        angle = random.uniform(-15, 15)
        h, w = img.shape
        M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
        img = cv2.warpAffine(img, M, (w, h), borderValue=0)
        mask = cv2.warpAffine(mask, M, (w, h), borderValue=0, flags=cv2.INTER_NEAREST)
    return img, mask

TRAIN_AUG = build_train_augmentation()

def compute_boundary_dist_map(mask01):
    posmask = mask01.astype(bool)
    if posmask.any() and (~posmask).any():
        pos_dist = distance_transform_edt(posmask)
        neg_dist = distance_transform_edt(~posmask)
        dist = neg_dist * (~posmask) - (pos_dist - 1) * posmask
    else:
        dist = np.zeros_like(mask01, dtype=np.float32)
    return dist.astype(np.float32)

class PlaqueDataset(Dataset):
    def __init__(self, image_paths, mask_paths, augment=False, return_dist_map=False):
        self.image_paths = list(image_paths)
        self.mask_paths = list(mask_paths)
        self.augment = augment
        self.return_dist_map = return_dist_map

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = cv2.imread(str(self.image_paths[idx]), cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(str(self.mask_paths[idx]), cv2.IMREAD_GRAYSCALE)
        mask01 = (mask > 127).astype(np.uint8)

        if self.augment:
            if TRAIN_AUG is not None:
                out = TRAIN_AUG(image=img, mask=mask01)
                img, mask01 = out["image"], out["mask"]
            else:
                img, mask01 = _fallback_augment(img, mask01)

        img_t = torch.from_numpy(img.astype(np.float32) / 255.0).unsqueeze(0)
        mask_t = torch.from_numpy(mask01.astype(np.float32)).unsqueeze(0)

        if self.return_dist_map:
            dist_t = torch.from_numpy(compute_boundary_dist_map(mask01)).unsqueeze(0)
            return img_t, mask_t, dist_t
        return img_t, mask_t

def paths_from_manifest(manifest_df):
    return manifest_df["prep_image"].tolist(), manifest_df["prep_mask"].tolist()

train_img_paths, train_mask_paths = paths_from_manifest(train_manifest)
test_img_paths, test_mask_paths = paths_from_manifest(test_manifest)

has_plaque = np.array([cv2.imread(p, cv2.IMREAD_GRAYSCALE).max() > 127 for p in train_mask_paths])
test_has_plaque = np.array([cv2.imread(p, cv2.IMREAD_GRAYSCALE).max() > 127 for p in test_mask_paths])
print(f"{len(train_img_paths)} training images ({has_plaque.sum()} with plaque) / "
      f"{len(test_img_paths)} test images ({test_has_plaque.sum()} with plaque) ready.")

493 training images (316 with plaque) / 48 test images (24 with plaque) ready.


## 5. Loss functions (identical weights/terms to FAB-UNet: `POS_WEIGHT`, Focal-Tversky, ramped boundary loss)

In [8]:
def dice_loss(logits, targets, eps=1.0):
    p = torch.sigmoid(logits.float()).reshape(-1)
    t = targets.float().reshape(-1)
    return 1 - (2 * (p * t).sum() + eps) / (p.sum() + t.sum() + eps)

def focal_tversky_loss(logits, targets, alpha=0.3, beta=0.7, gamma=0.75, eps=1.0):
    p = torch.sigmoid(logits.float()).reshape(-1)
    t = targets.float().reshape(-1)
    tp = (p * t).sum()
    fp = (p * (1 - t)).sum()
    fn = ((1 - p) * t).sum()
    tv = (tp + eps) / (tp + alpha * fp + beta * fn + eps)
    return (1 - tv) ** gamma

def boundary_loss(logits, dist_map):
    d = dist_map.float()
    d = d / (d.abs().amax(dim=(1, 2, 3), keepdim=True) + 1e-6)
    return (torch.sigmoid(logits.float()) * d).mean()

BND_MAX = 1.0
BND_WEIGHT = {"w": 0.0}
POS_WEIGHT = 5.0

def combined_loss(outputs, target, dist_map=None, w_bce=0.3, w_dice=0.3, w_ft=0.4):
    main = outputs["main"].float()
    target = target.float()
    pos_w = torch.tensor(POS_WEIGHT, device=main.device)
    loss = (w_bce * F.binary_cross_entropy_with_logits(main, target, pos_weight=pos_w)
            + w_dice * dice_loss(main, target)
            + w_ft * focal_tversky_loss(main, target))
    if dist_map is not None and BND_WEIGHT["w"] > 0:
        loss = loss + BND_WEIGHT["w"] * boundary_loss(main, dist_map)
    for key, w in (("aux3", 0.3), ("aux2", 0.2)):
        if key in outputs:
            aux = outputs[key].float()
            loss = loss + w * (F.binary_cross_entropy_with_logits(aux, target, pos_weight=pos_w)
                               + dice_loss(aux, target))
    return loss

## 6. Metrics (pixel-level confusion matrix + per-image Dice, incl. the paper-comparable `plaque_image_dice`)

In [9]:
class ConfusionAccumulator:
    def __init__(self):
        self.tp = self.fp = self.tn = self.fn = 0

    def update(self, pred_bin, target_bin):
        pred_bin = pred_bin.astype(bool).ravel()
        target_bin = target_bin.astype(bool).ravel()
        self.tp += int(np.sum(pred_bin & target_bin))
        self.fp += int(np.sum(pred_bin & ~target_bin))
        self.fn += int(np.sum(~pred_bin & target_bin))
        self.tn += int(np.sum(~pred_bin & ~target_bin))

    def compute(self, eps=1e-9):
        precision = self.tp / (self.tp + self.fp + eps)
        recall = self.tp / (self.tp + self.fn + eps)
        f1 = 2 * precision * recall / (precision + recall + eps)
        iou = self.tp / (self.tp + self.fp + self.fn + eps)
        dice = 2 * self.tp / (2 * self.tp + self.fp + self.fn + eps)
        accuracy = (self.tp + self.tn) / (self.tp + self.tn + self.fp + self.fn + eps)
        specificity = self.tn / (self.tn + self.fp + eps)
        return dict(precision=precision, recall=recall, f1=f1, iou=iou, dice=dice,
                    accuracy=accuracy, specificity=specificity)

    def matrix(self):
        return np.array([[self.tn, self.fp], [self.fn, self.tp]])


def image_dice(pred, gt):
    s = pred.sum() + gt.sum()
    return 1.0 if s == 0 else 2.0 * (pred & gt).sum() / s


@torch.no_grad()
def evaluate_loader(model, loader, device, threshold=0.5, mc_dropout=False):
    model.eval()
    acc = ConfusionAccumulator()
    val_loss_total, n_batches = 0.0, 0
    img_dice, img_has = [], []
    for batch in loader:
        if len(batch) == 3:
            imgs, masks, dists = batch
            dists = dists.to(device)
        else:
            imgs, masks = batch
            dists = None
        imgs, masks = imgs.to(device), masks.to(device)
        outputs = model(imgs, mc_dropout=mc_dropout)
        loss = combined_loss(outputs, masks, dists)
        val_loss_total += loss.item(); n_batches += 1
        probs = torch.sigmoid(outputs["main"].float()).cpu().numpy()
        preds = (probs > threshold)
        tgts = masks.cpu().numpy() > 0.5
        acc.update(preds, tgts)
        for p, t in zip(preds, tgts):
            img_dice.append(image_dice(p, t)); img_has.append(bool(t.any()))
    metrics = acc.compute()
    metrics["loss"] = val_loss_total / max(n_batches, 1)
    img_dice, img_has = np.array(img_dice), np.array(img_has)
    metrics["mean_image_dice"] = float(img_dice.mean())
    metrics["plaque_image_dice"] = float(img_dice[img_has].mean()) if img_has.any() else float("nan")
    return metrics, acc

## 7. Training utilities (identical `EarlyStopping`/`train_one_fold`, checkpoint-selection on `val_dice`)

In [10]:
class EarlyStopping:
    def __init__(self, patience=10, mode="min"):
        self.patience, self.mode = patience, mode
        self.best, self.count, self.should_stop = None, 0, False

    def step(self, value):
        better = (self.best is None or (self.mode == "min" and value < self.best)
                  or (self.mode == "max" and value > self.best))
        if better:
            self.best, self.count = value, 0
            return True
        self.count += 1
        if self.count >= self.patience:
            self.should_stop = True
        return False


def train_one_fold(model, train_loader, val_loader, device, epochs=CFG.EPOCHS, lr=CFG.LR,
                   patience=CFG.EARLY_STOP_PATIENCE, tag="fold", use_boundary=True, verbose=True):
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    warm = 3
    def lr_lambda(e):
        if e < warm:
            return (e + 1) / warm
        prog = (e - warm) / max(1, epochs - warm)
        return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * prog))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
    stopper = EarlyStopping(patience=patience, mode="max")

    history = {"train_loss": [], "val_loss": [], "val_dice": [], "val_iou": [], "val_f1": []}
    best_state, best_epoch = None, 0

    for epoch in range(1, epochs + 1):
        BND_WEIGHT["w"] = BND_MAX * min(1.0, epoch / (0.4 * epochs)) if use_boundary else 0.0
        model.train()
        running_loss, n_batches = 0.0, 0
        pbar = tqdm(train_loader, desc=f"[{tag}] epoch {epoch}/{epochs}", leave=False)
        for imgs, masks, dists in pbar:
            imgs, masks, dists = imgs.to(device), masks.to(device), dists.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                outputs = model(imgs)
            loss = combined_loss(outputs, masks, dists)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item(); n_batches += 1
            pbar.set_postfix(loss=running_loss / n_batches)

        train_loss = running_loss / max(n_batches, 1)
        val_metrics, _ = evaluate_loader(model, val_loader, device)
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_metrics["loss"])
        history["val_dice"].append(val_metrics["dice"])
        history["val_iou"].append(val_metrics["iou"])
        history["val_f1"].append(val_metrics["f1"])

        if stopper.step(val_metrics["dice"]):
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = epoch

        if verbose and (epoch <= 3 or epoch % 5 == 0 or stopper.count == 0):
            print(f"[{tag}] epoch {epoch:03d} | lr {optimizer.param_groups[0]['lr']:.2e} | "
                  f"train_loss {train_loss:.4f} | val_loss {val_metrics['loss']:.4f} | "
                  f"val_dice {val_metrics['dice']:.4f} | val_iou {val_metrics['iou']:.4f}")

        if stopper.should_stop:
            print(f"[{tag}] Early stopping at epoch {epoch} (best val_dice={stopper.best:.4f} @ epoch {best_epoch})")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    history["best_epoch"] = best_epoch
    BND_WEIGHT["w"] = 0.0
    return model, history

## 8. Architectures: Attention U-Net, U-Net++, ResUNet++

In [11]:
# ============================================================================
# Shared conv block used by all three Stage-1 architectures
# ============================================================================
class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)


# ============================================================================
# 1. ATTENTION U-NET (Oktay et al., 2018)
#    Plain U-Net encoder/decoder; additive attention gates filter each skip
#    connection using the coarser decoder signal before concatenation.
# ============================================================================
class AttentionGate(nn.Module):
    def __init__(self, c_g, c_x, c_mid):
        super().__init__()
        self.wg = nn.Sequential(nn.Conv2d(c_g, c_mid, 1), nn.BatchNorm2d(c_mid))
        self.wx = nn.Sequential(nn.Conv2d(c_x, c_mid, 1), nn.BatchNorm2d(c_mid))
        self.psi = nn.Sequential(nn.Conv2d(c_mid, 1, 1), nn.BatchNorm2d(1), nn.Sigmoid())
    def forward(self, g, x):
        g1 = self.wg(g); x1 = self.wx(x)
        if g1.shape[-2:] != x1.shape[-2:]:
            g1 = F.interpolate(g1, size=x1.shape[-2:], mode="bilinear", align_corners=False)
        psi = self.psi(F.relu(g1 + x1, inplace=True))
        return x * psi

class AttentionUNet(nn.Module):
    def __init__(self, in_ch=1, base=32, mc_dropout_p=0.0):
        super().__init__()
        ch = [base, base*2, base*4, base*8, base*16]
        self.pool = nn.MaxPool2d(2)
        self.enc1 = ConvBlock(in_ch, ch[0]); self.enc2 = ConvBlock(ch[0], ch[1])
        self.enc3 = ConvBlock(ch[1], ch[2]); self.enc4 = ConvBlock(ch[2], ch[3])
        self.bottleneck = ConvBlock(ch[3], ch[4])
        self.up4 = nn.ConvTranspose2d(ch[4], ch[3], 2, stride=2); self.ag4 = AttentionGate(ch[3], ch[3], ch[3]//2); self.dec4 = ConvBlock(ch[3]*2, ch[3])
        self.up3 = nn.ConvTranspose2d(ch[3], ch[2], 2, stride=2); self.ag3 = AttentionGate(ch[2], ch[2], ch[2]//2); self.dec3 = ConvBlock(ch[2]*2, ch[2])
        self.up2 = nn.ConvTranspose2d(ch[2], ch[1], 2, stride=2); self.ag2 = AttentionGate(ch[1], ch[1], ch[1]//2); self.dec2 = ConvBlock(ch[1]*2, ch[1])
        self.up1 = nn.ConvTranspose2d(ch[1], ch[0], 2, stride=2); self.ag1 = AttentionGate(ch[0], ch[0], ch[0]//2); self.dec1 = ConvBlock(ch[0]*2, ch[0])
        self.drop = nn.Dropout2d(mc_dropout_p) if mc_dropout_p > 0 else nn.Identity()
        self.head = nn.Conv2d(ch[0], 1, 1)

    def forward(self, x, mc_dropout=False):
        e1 = self.enc1(x); e2 = self.enc2(self.pool(e1)); e3 = self.enc3(self.pool(e2)); e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        d4 = self.up4(b);  d4 = self.dec4(torch.cat([self.ag4(d4, e4), d4], dim=1))
        d3 = self.up3(d4); d3 = self.dec3(torch.cat([self.ag3(d3, e3), d3], dim=1))
        d2 = self.up2(d3); d2 = self.dec2(torch.cat([self.ag2(d2, e2), d2], dim=1))
        d1 = self.up1(d2); d1 = self.dec1(torch.cat([self.ag1(d1, e1), d1], dim=1))
        if mc_dropout: self.drop.train()
        return {"main": self.head(self.drop(d1))}


# ============================================================================
# 2. U-NET++ / NESTED U-NET (Zhou et al., 2018)
#    Dense nested skip pathways X^{i,j}; final output from the deepest (x0_4) node.
# ============================================================================
class UNetPlusPlus(nn.Module):
    def __init__(self, in_ch=1, base=32, mc_dropout_p=0.0):
        super().__init__()
        c = [base, base*2, base*4, base*8, base*16]
        self.pool = nn.MaxPool2d(2)
        self.up = lambda t, ref: F.interpolate(t, size=ref.shape[-2:], mode="bilinear", align_corners=False)

        self.conv0_0 = ConvBlock(in_ch, c[0]); self.conv1_0 = ConvBlock(c[0], c[1])
        self.conv2_0 = ConvBlock(c[1], c[2]);  self.conv3_0 = ConvBlock(c[2], c[3]); self.conv4_0 = ConvBlock(c[3], c[4])

        self.conv0_1 = ConvBlock(c[0]+c[1], c[0]); self.conv1_1 = ConvBlock(c[1]+c[2], c[1])
        self.conv2_1 = ConvBlock(c[2]+c[3], c[2]); self.conv3_1 = ConvBlock(c[3]+c[4], c[3])

        self.conv0_2 = ConvBlock(c[0]*2+c[1], c[0]); self.conv1_2 = ConvBlock(c[1]*2+c[2], c[1])
        self.conv2_2 = ConvBlock(c[2]*2+c[3], c[2])

        self.conv0_3 = ConvBlock(c[0]*3+c[1], c[0]); self.conv1_3 = ConvBlock(c[1]*3+c[2], c[1])
        self.conv0_4 = ConvBlock(c[0]*4+c[1], c[0])

        self.drop = nn.Dropout2d(mc_dropout_p) if mc_dropout_p > 0 else nn.Identity()
        self.head = nn.Conv2d(c[0], 1, 1)

    def forward(self, x, mc_dropout=False):
        x0_0 = self.conv0_0(x)
        x1_0 = self.conv1_0(self.pool(x0_0))
        x0_1 = self.conv0_1(torch.cat([x0_0, self.up(x1_0, x0_0)], 1))

        x2_0 = self.conv2_0(self.pool(x1_0))
        x1_1 = self.conv1_1(torch.cat([x1_0, self.up(x2_0, x1_0)], 1))
        x0_2 = self.conv0_2(torch.cat([x0_0, x0_1, self.up(x1_1, x0_0)], 1))

        x3_0 = self.conv3_0(self.pool(x2_0))
        x2_1 = self.conv2_1(torch.cat([x2_0, self.up(x3_0, x2_0)], 1))
        x1_2 = self.conv1_2(torch.cat([x1_0, x1_1, self.up(x2_1, x1_0)], 1))
        x0_3 = self.conv0_3(torch.cat([x0_0, x0_1, x0_2, self.up(x1_2, x0_0)], 1))

        x4_0 = self.conv4_0(self.pool(x3_0))
        x3_1 = self.conv3_1(torch.cat([x3_0, self.up(x4_0, x3_0)], 1))
        x2_2 = self.conv2_2(torch.cat([x2_0, x2_1, self.up(x3_1, x2_0)], 1))
        x1_3 = self.conv1_3(torch.cat([x1_0, x1_1, x1_2, self.up(x2_2, x1_0)], 1))
        x0_4 = self.conv0_4(torch.cat([x0_0, x0_1, x0_2, x0_3, self.up(x1_3, x0_0)], 1))

        if mc_dropout: self.drop.train()
        return {"main": self.head(self.drop(x0_4))}


# ============================================================================
# 3. RESUNET++ (Jha et al., 2019)
#    Residual encoder blocks with squeeze-and-excitation, an ASPP bridge,
#    and attention-gated residual decoder blocks.
# ============================================================================
class SEBlock(nn.Module):
    def __init__(self, c, r=8):
        super().__init__()
        self.fc = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(c, max(c//r,4), 1), nn.ReLU(inplace=True),
                                 nn.Conv2d(max(c//r,4), c, 1), nn.Sigmoid())
    def forward(self, x): return x * self.fc(x)

class ResBlock(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(cin); self.conv1 = nn.Conv2d(cin, cout, 3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(cout); self.conv2 = nn.Conv2d(cout, cout, 3, padding=1, bias=False)
        self.se = SEBlock(cout)
        self.skip = nn.Conv2d(cin, cout, 1, stride=stride) if (cin != cout or stride != 1) else nn.Identity()
    def forward(self, x):
        s = self.skip(x)
        h = self.conv1(F.relu(self.bn1(x), inplace=True))
        h = self.conv2(F.relu(self.bn2(h), inplace=True))
        h = self.se(h)
        return h + s

class ASPP(nn.Module):
    def __init__(self, c, rates=(1, 6, 12, 18)):
        super().__init__()
        self.branches = nn.ModuleList([
            nn.Sequential(nn.Conv2d(c, c, 3, padding=r, dilation=r, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True))
            for r in rates
        ])
        self.proj = nn.Sequential(nn.Conv2d(c*len(rates), c, 1), nn.BatchNorm2d(c), nn.ReLU(inplace=True))
    def forward(self, x):
        return self.proj(torch.cat([b(x) for b in self.branches], dim=1))

class ResUNetPlusPlus(nn.Module):
    def __init__(self, in_ch=1, base=32, mc_dropout_p=0.0):
        super().__init__()
        c = [base, base*2, base*4, base*8]
        self.stem = nn.Sequential(nn.Conv2d(in_ch, c[0], 3, padding=1, bias=False), nn.BatchNorm2d(c[0]), nn.ReLU(inplace=True),
                                   nn.Conv2d(c[0], c[0], 3, padding=1, bias=False))
        self.stem_skip = nn.Conv2d(in_ch, c[0], 1)
        self.enc2 = ResBlock(c[0], c[1], stride=2)
        self.enc3 = ResBlock(c[1], c[2], stride=2)
        self.enc4 = ResBlock(c[2], c[3], stride=2)
        self.bridge = ASPP(c[3])

        self.ag3 = AttentionGate(c[3], c[2], c[2]//2); self.dec3 = ResBlock(c[3]+c[2], c[2])
        self.ag2 = AttentionGate(c[2], c[1], c[1]//2); self.dec2 = ResBlock(c[2]+c[1], c[1])
        self.ag1 = AttentionGate(c[1], c[0], c[0]//2); self.dec1 = ResBlock(c[1]+c[0], c[0])

        self.drop = nn.Dropout2d(mc_dropout_p) if mc_dropout_p > 0 else nn.Identity()
        self.head = nn.Conv2d(c[0], 1, 1)

    def _up_to(self, t, ref):
        return F.interpolate(t, size=ref.shape[-2:], mode="bilinear", align_corners=False)

    def forward(self, x, mc_dropout=False):
        e1 = self.stem(x) + self.stem_skip(x)
        e2 = self.enc2(e1); e3 = self.enc3(e2); e4 = self.enc4(e3)
        b = self.bridge(e4)

        d3 = self._up_to(b, e3);  d3 = self.dec3(torch.cat([self.ag3(d3, e3), d3], dim=1))
        d2 = self._up_to(d3, e2); d2 = self.dec2(torch.cat([self.ag2(d2, e2), d2], dim=1))
        d1 = self._up_to(d2, e1); d1 = self.dec1(torch.cat([self.ag1(d1, e1), d1], dim=1))

        if mc_dropout: self.drop.train()
        out_small = self.head(self.drop(d1))
        return {"main": self._up_to(out_small, x)}


print("Architectures defined: AttentionUNet, UNetPlusPlus, ResUNetPlusPlus")
for name, M in [("AttentionUNet", AttentionUNet), ("UNetPlusPlus", UNetPlusPlus), ("ResUNetPlusPlus", ResUNetPlusPlus)]:
    n = sum(p.numel() for p in M(in_ch=1, base=32).parameters())
    print(f"  {name:18s} {n/1e6:6.2f} M params")

Architectures defined: AttentionUNet, UNetPlusPlus, ResUNetPlusPlus
  AttentionUNet        7.85 M params
  UNetPlusPlus         9.16 M params
  ResUNetPlusPlus      4.74 M params


## 9. Train/val/test split (same seed + stratification as FAB-UNet's `final_model` split, for a fair comparison)

In [12]:
set_seed(SEED)
final_tr_idx, final_va_idx = train_test_split(np.arange(len(train_img_paths)), test_size=0.1,
                                                random_state=SEED, shuffle=True, stratify=has_plaque)

final_tr_imgs = [train_img_paths[i] for i in final_tr_idx]; final_tr_masks = [train_mask_paths[i] for i in final_tr_idx]
final_va_imgs = [train_img_paths[i] for i in final_va_idx]; final_va_masks = [train_mask_paths[i] for i in final_va_idx]

final_train_ds = PlaqueDataset(final_tr_imgs, final_tr_masks, augment=True, return_dist_map=True)
final_val_ds = PlaqueDataset(final_va_imgs, final_va_masks, augment=False, return_dist_map=True)
final_train_loader = DataLoader(final_train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True,
                                  num_workers=CFG.NUM_WORKERS, drop_last=True)
final_val_loader = DataLoader(final_val_ds, batch_size=CFG.BATCH_SIZE, shuffle=False,
                                num_workers=CFG.NUM_WORKERS)

test_ds = PlaqueDataset(test_img_paths, test_mask_paths, augment=False, return_dist_map=True)
test_loader = DataLoader(test_ds, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=CFG.NUM_WORKERS)

print(f"Split: {len(final_tr_imgs)} train / {len(final_va_imgs)} val / {len(test_img_paths)} test "
      f"(identical split seed/protocol to the FAB-UNet notebook's final_model, for a fair comparison)")

Split: 443 train / 50 val / 48 test (identical split seed/protocol to the FAB-UNet notebook's final_model, for a fair comparison)


## 10. Train and evaluate each baseline

In [13]:
RESULTS = {}         # model_name -> test metrics dict
PARAM_COUNTS = {}     # model_name -> parameter count

In [14]:
# ---------------------------------------------------------------------------
# attention_unet
# ---------------------------------------------------------------------------
print("="*70); print("Training: attention_unet"); print("="*70)

set_seed(SEED)
model = AttentionUNet(in_ch=1, base=32, mc_dropout_p=CFG.MC_DROPOUT_P)
model, history_attention_unet = train_one_fold(model, final_train_loader, final_val_loader, DEVICE,
                                              epochs=CFG.EPOCHS, tag="attention_unet")
torch.save(model.state_dict(), f"{CFG.OUTPUT_DIR}/attention_unet_final_best.pth")

test_metrics_attention_unet, test_acc_attention_unet = evaluate_loader(model, test_loader, DEVICE)
print("\nTest-set metrics (attention_unet):")
for k, v in test_metrics_attention_unet.items():
    print(f"  {k:18s}: {v:.4f}")

RESULTS["attention_unet"] = test_metrics_attention_unet
n_params_attention_unet = sum(p.numel() for p in model.parameters())
PARAM_COUNTS["attention_unet"] = n_params_attention_unet
print(f"Parameters: {n_params_attention_unet/1e6:.2f} M")

del model
torch.cuda.empty_cache()

Training: attention_unet


[attention_unet] epoch 1/120:   0%|          | 0/73 [00:00<?, ?it/s]

[attention_unet] epoch 001 | lr 6.67e-04 | train_loss 0.8257 | val_loss 0.7825 | val_dice 0.0212 | val_iou 0.0107


[attention_unet] epoch 2/120:   0%|          | 0/73 [00:00<?, ?it/s]

[attention_unet] epoch 002 | lr 1.00e-03 | train_loss 0.7765 | val_loss 0.7796 | val_dice 0.1009 | val_iou 0.0532


[attention_unet] epoch 3/120:   0%|          | 0/73 [00:00<?, ?it/s]

... (137 outputs left out)

[attention_unet] epoch 98/120:   0%|          | 0/73 [00:00<?, ?it/s]

[attention_unet] epoch 99/120:   0%|          | 0/73 [00:00<?, ?it/s]

[attention_unet] epoch 100/120:   0%|          | 0/73 [00:00<?, ?it/s]

[attention_unet] epoch 100 | lr 7.97e-05 | train_loss 0.2691 | val_loss 0.4689 | val_dice 0.5247 | val_iou 0.3557
[attention_unet] Early stopping at epoch 100 (best val_dice=0.5651 @ epoch 80)

Test-set metrics (attention_unet):
  precision         : 0.5904
  recall            : 0.6496
  f1                : 0.6186
  iou               : 0.4478
  dice              : 0.6186
  accuracy          : 0.9931
  specificity       : 0.9961
  loss              : 0.4899
  mean_image_dice   : 0.5594
  plaque_image_dice : 0.5771
Parameters: 7.85 M


In [15]:
# ---------------------------------------------------------------------------
# unet_plusplus
# ---------------------------------------------------------------------------
print("="*70); print("Training: unet_plusplus"); print("="*70)

set_seed(SEED)
model = UNetPlusPlus(in_ch=1, base=32, mc_dropout_p=CFG.MC_DROPOUT_P)
model, history_unet_plusplus = train_one_fold(model, final_train_loader, final_val_loader, DEVICE,
                                              epochs=CFG.EPOCHS, tag="unet_plusplus")
torch.save(model.state_dict(), f"{CFG.OUTPUT_DIR}/unet_plusplus_final_best.pth")

test_metrics_unet_plusplus, test_acc_unet_plusplus = evaluate_loader(model, test_loader, DEVICE)
print("\nTest-set metrics (unet_plusplus):")
for k, v in test_metrics_unet_plusplus.items():
    print(f"  {k:18s}: {v:.4f}")

RESULTS["unet_plusplus"] = test_metrics_unet_plusplus
n_params_unet_plusplus = sum(p.numel() for p in model.parameters())
PARAM_COUNTS["unet_plusplus"] = n_params_unet_plusplus
print(f"Parameters: {n_params_unet_plusplus/1e6:.2f} M")

del model
torch.cuda.empty_cache()

Training: unet_plusplus


[unet_plusplus] epoch 1/120:   0%|          | 0/73 [00:00<?, ?it/s]

[unet_plusplus] epoch 001 | lr 6.67e-04 | train_loss 0.8548 | val_loss 0.8187 | val_dice 0.1502 | val_iou 0.0812


[unet_plusplus] epoch 2/120:   0%|          | 0/73 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x78e1f84dbf60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    Exception ignored in: self._shutdown_workers()
<function _MultiProcessingDataLoaderIter.__del__ at 0x78e1f84dbf60>  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers

Traceback (most recent call last):
      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
if w.is_alive():    self._shutdown_workers()

   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
      if w.is_alive():
        ^  ^ ^^^^^^^^^^^^^^^^^^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    ^assert self._parent_pid == os.getpid(), 'can only test a child process'

  File "/usr/lib/python

[unet_plusplus] epoch 002 | lr 1.00e-03 | train_loss 0.8082 | val_loss 0.8405 | val_dice 0.1006 | val_iou 0.0530


... (139 outputs left out)

[unet_plusplus] epoch 100/120:   0%|          | 0/73 [00:00<?, ?it/s]

[unet_plusplus] epoch 100 | lr 7.97e-05 | train_loss 0.2715 | val_loss 0.4737 | val_dice 0.5333 | val_iou 0.3636


[unet_plusplus] epoch 101/120:   0%|          | 0/73 [00:00<?, ?it/s]

[unet_plusplus] Early stopping at epoch 101 (best val_dice=0.5638 @ epoch 81)

Test-set metrics (unet_plusplus):
  precision         : 0.6254
  recall            : 0.6701
  f1                : 0.6470
  iou               : 0.4782
  dice              : 0.6470
  accuracy          : 0.9937
  specificity       : 0.9965
  loss              : 0.4797
  mean_image_dice   : 0.5730
  plaque_image_dice : 0.6043
Parameters: 9.16 M


In [16]:
# ---------------------------------------------------------------------------
# resunet_plusplus
# ---------------------------------------------------------------------------
print("="*70); print("Training: resunet_plusplus"); print("="*70)

set_seed(SEED)
model = ResUNetPlusPlus(in_ch=1, base=32, mc_dropout_p=CFG.MC_DROPOUT_P)
model, history_resunet_plusplus = train_one_fold(model, final_train_loader, final_val_loader, DEVICE,
                                              epochs=CFG.EPOCHS, tag="resunet_plusplus")
torch.save(model.state_dict(), f"{CFG.OUTPUT_DIR}/resunet_plusplus_final_best.pth")

test_metrics_resunet_plusplus, test_acc_resunet_plusplus = evaluate_loader(model, test_loader, DEVICE)
print("\nTest-set metrics (resunet_plusplus):")
for k, v in test_metrics_resunet_plusplus.items():
    print(f"  {k:18s}: {v:.4f}")

RESULTS["resunet_plusplus"] = test_metrics_resunet_plusplus
n_params_resunet_plusplus = sum(p.numel() for p in model.parameters())
PARAM_COUNTS["resunet_plusplus"] = n_params_resunet_plusplus
print(f"Parameters: {n_params_resunet_plusplus/1e6:.2f} M")

del model
torch.cuda.empty_cache()

Training: resunet_plusplus


[resunet_plusplus] epoch 1/120:   0%|          | 0/73 [00:00<?, ?it/s]

[resunet_plusplus] epoch 001 | lr 6.67e-04 | train_loss 0.6351 | val_loss 0.5712 | val_dice 0.2975 | val_iou 0.1747


[resunet_plusplus] epoch 2/120:   0%|          | 0/73 [00:00<?, ?it/s]

[resunet_plusplus] epoch 002 | lr 1.00e-03 | train_loss 0.5593 | val_loss 0.5473 | val_dice 0.3500 | val_iou 0.2122


[resunet_plusplus] epoch 3/120:   0%|          | 0/73 [00:00<?, ?it/s]

... (83 outputs left out)

[resunet_plusplus] epoch 58/120:   0%|          | 0/73 [00:00<?, ?it/s]

[resunet_plusplus] epoch 59/120:   0%|          | 0/73 [00:00<?, ?it/s]

[resunet_plusplus] epoch 60/120:   0%|          | 0/73 [00:00<?, ?it/s]

[resunet_plusplus] epoch 060 | lr 5.25e-04 | train_loss 0.2760 | val_loss 0.4788 | val_dice 0.5191 | val_iou 0.3505
[resunet_plusplus] Early stopping at epoch 60 (best val_dice=0.5378 @ epoch 40)

Test-set metrics (resunet_plusplus):
  precision         : 0.4781
  recall            : 0.6463
  f1                : 0.5496
  iou               : 0.3789
  dice              : 0.5496
  accuracy          : 0.9909
  specificity       : 0.9939
  loss              : 0.5137
  mean_image_dice   : 0.3743
  plaque_image_dice : 0.5403
Parameters: 4.74 M


## 11. Comparison table

**Fill in your own FAB-UNet test-set numbers below** (from your most recent FAB-UNet notebook
run) so the table includes it for context, without needing to retrain FAB-UNet in this
notebook. Replace every `None` with the printed value from that run's
`Test-set metrics (FAB-UNet):` block.

In [17]:
# <-- EDIT THESE: paste your own FAB-UNet (and, if you have it, baseline U-Net) test metrics here
FABUNET_REFERENCE = {
    "precision": None, "recall": None, "f1": None, "iou": None, "dice": None,
    "accuracy": None, "specificity": None, "mean_image_dice": None, "plaque_image_dice": None,
}
if all(v is not None for v in FABUNET_REFERENCE.values()):
    RESULTS["FAB-UNet (reference, from prior run)"] = FABUNET_REFERENCE
    PARAM_COUNTS["FAB-UNet (reference, from prior run)"] = float("nan")
else:
    print("FABUNET_REFERENCE not filled in -- skipping it in the comparison table. "
          "Edit the dict above with your FAB-UNet numbers to include it.")

FABUNET_REFERENCE not filled in -- skipping it in the comparison table. Edit the dict above with your FAB-UNet numbers to include it.


In [18]:
# ---------------------------------------------------------------------------
# Comparison table
# ---------------------------------------------------------------------------
rows = []
for name, m in RESULTS.items():
    rows.append({
        "model": name,
        "precision": m["precision"], "recall": m["recall"], "f1": m["f1"],
        "iou": m["iou"], "dice": m["dice"], "accuracy": m["accuracy"],
        "specificity": m["specificity"], "mean_image_dice": m["mean_image_dice"],
        "plaque_image_dice": m["plaque_image_dice"],
        "params_M": PARAM_COUNTS.get(name, float("nan")) / 1e6,
    })



comparison_df = pd.DataFrame(rows).set_index("model")
comparison_df.to_csv(f"{CFG.OUTPUT_DIR}/stage1_comparison.csv")
print(f"Saved comparison table to {CFG.OUTPUT_DIR}/stage1_comparison.csv")
print()
print("NOTE: the metric that is directly comparable to the Rulloni et al. (2026) dataset-paper")
print("baseline of 0.62 Dice is 'plaque_image_dice' (mean per-image Dice on images that")
print("actually contain a plaque) -- NOT the pooled 'dice' column, which aggregates every")
print("pixel across the whole test set (including easy plaque-free images) and is not the")
print("same metric, even though the two numbers can look superficially similar.")
print()
comparison_df.round(4)

Saved comparison table to /kaggle/working/outputs_stage1/stage1_comparison.csv

NOTE: the metric that is directly comparable to the Rulloni et al. (2026) dataset-paper
baseline of 0.62 Dice is 'plaque_image_dice' (mean per-image Dice on images that
actually contain a plaque) -- NOT the pooled 'dice' column, which aggregates every
pixel across the whole test set (including easy plaque-free images) and is not the
same metric, even though the two numbers can look superficially similar.



,precision,recall,f1,iou,dice,accuracy,specificity,mean_image_dice,plaque_image_dice,params_M
model,,,,,,,,,,
attention_unet,0.5904,0.6496,0.6186,0.4478,0.6186,0.9931,0.9961,0.5594,0.5771,7.8512
unet_plusplus,0.6254,0.6701,0.6470,0.4782,0.6470,0.9937,0.9965,0.5730,0.6043,9.1591
resunet_plusplus,0.4781,0.6463,0.5496,0.3789,0.5496,0.9909,0.9939,0.3743,0.5403,4.7413


## Conclusion

This notebook isolates **architecture choice alone**: every baseline here was trained with the
identical loss, optimizer schedule, checkpoint-selection rule, and data split as FAB-UNet.
If FAB-UNet's `plaque_image_dice` beats all three of these, that is evidence its specific
combination of spectral attention + dual-branch uncertainty + boundary-aware loss is doing
real work beyond what generic attention gates (Attention U-Net), dense nested skips (U-Net++),
or a residual+SE+ASPP design (ResUNet++) already provide. If one of these baselines matches or
beats FAB-UNet, that narrows down **which** specific mechanism is or isn't pulling its weight.

Remember: compare on **`plaque_image_dice`**, the column that matches the Rulloni et al. (2026)
dataset paper's own reporting convention -- not the pooled `dice` column.